# Anomalous Authentication Behavior Detection
## AI/ML Capstone Project — Daisy Wu

This notebook walks through the complete analysis pipeline for detecting anomalous
authentication behavior in enterprise security logs using the **LANL Cyber Security Dataset**.

### Pipeline Overview

1. Data Loading & Preprocessing
2. Exploratory Data Analysis
3. Feature Engineering (Module 8)
4. PCA & Clustering (Module 6)
5. Time Series Analysis (Module 10)
6. Model Selection & Regularization (Module 9 / Module 15)
7. Classification: KNN & Logistic Regression (Modules 11 & 13)
8. Decision Trees (Module 14)
9. Final Comparison & Conclusions

## 1. Setup and Imports

In [ ]:
import sys
sys.path.insert(0, '..')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

from src.data_preprocessing import (
    load_auth_data, load_redteam_labels, clean_data,
    label_redteam_events, save_processed_data
)
from src.feature_engineering import build_feature_matrix, FEATURE_COLUMNS, TARGET_COLUMN
from src.eda import (
    generate_summary_statistics, run_full_eda,
    plot_login_distribution, plot_success_failure_ratio,
    plot_hourly_pattern, plot_top_users,
    plot_auth_type_distribution, plot_redteam_timeline,
    plot_correlation_matrix
)
from src.clustering_pca import (
    apply_pca, plot_explained_variance,
    find_optimal_clusters, apply_kmeans, plot_pca_clusters
)
from src.time_series import (
    aggregate_login_timeseries, decompose_timeseries,
    detect_temporal_anomalies, plot_timeseries_with_anomalies
)
from src.model_selection import (
    compare_models, tune_regularization,
    plot_learning_curve, evaluate_model, save_model
)
from src.classification import (
    train_knn, find_optimal_k, train_logistic_regression,
    plot_roc_curves, plot_confusion_matrices, print_classification_report
)
from src.decision_trees import (
    train_decision_tree, tune_tree_depth,
    visualize_tree, extract_rules, get_feature_importance
)
from src.utils import load_processed_data, prepare_train_test, print_section

%matplotlib inline
plt.rcParams['figure.dpi'] = 100
pd.set_option('display.max_columns', 50)

print('All imports successful!')

## 2. Data Loading and Preprocessing

The LANL dataset contains ~1.6 billion authentication events over 58 days.
We start with a sample for development, then scale up.

**Before running:** Download the LANL dataset and place `auth.txt.gz` and `redteam.txt.gz` in `data/raw/`.

In [ ]:
# Load a sample of authentication data
# Adjust nrows as needed: 1M rows is ~1% of the full dataset
SAMPLE_SIZE = 1_000_000

df_raw = load_auth_data(nrows=SAMPLE_SIZE)
df_raw.head(10)

In [ ]:
# Load red team labels (known malicious events)
redteam = load_redteam_labels()
print(f"\nRed team event sample:")
redteam.head()

In [ ]:
# Clean the data: parse user@domain, convert success/failure, add time features
df = clean_data(df_raw)
df.head()

In [ ]:
# Label red team events as suspicious
df = label_redteam_events(df, redteam)

# Save cleaned data for reuse
save_processed_data(df, 'auth_cleaned.parquet')

## 3. Exploratory Data Analysis

In [ ]:
# Summary statistics
generate_summary_statistics(df)

In [ ]:
# Login volume over time
plot_login_distribution(df)

In [ ]:
# Success vs. failure breakdown
plot_success_failure_ratio(df)

In [ ]:
# Hourly login pattern
plot_hourly_pattern(df)

In [ ]:
# Most active users
plot_top_users(df, top_n=20)

In [ ]:
# Authentication and logon type distributions
plot_auth_type_distribution(df)
from src.eda import plot_logon_type_distribution
plot_logon_type_distribution(df)

In [ ]:
# Red team events timeline
plot_redteam_timeline(df)

## 4. Feature Engineering (Module 8)

Engineer behavioral features from raw log fields:

- **Temporal**: hour, day, off-hours flag, weekend flag, time since last login
- **Frequency**: login counts over 1h/6h/24h windows, daily failure ratio
- **Diversity**: distinct computers, auth types, logon types per user per day
- **Baseline**: total events, mean daily events, new destination flag

In [ ]:
print("Building feature matrix (this may take a few minutes)...")
df_features = build_feature_matrix(df)
df_features.head(10)

In [ ]:
# Save feature matrix for reuse
save_processed_data(df_features, 'auth_features.parquet')

print(f"\nFeature columns: {[c for c in df_features.columns if c != TARGET_COLUMN]}")
print(f"Target column: {TARGET_COLUMN}")
print(f"\nClass distribution:")
print(df_features[TARGET_COLUMN].value_counts())

In [ ]:
# Correlation matrix of engineered features
plot_correlation_matrix(df_features)

## 5. PCA and Clustering (Module 6)

Use PCA to reduce the high-dimensional feature space for visualization.
Apply K-Means clustering to discover natural groupings of login behavior.

In [ ]:
# Separate features and target
X_all = df_features.drop(columns=[TARGET_COLUMN]).values
y_all = df_features[TARGET_COLUMN].values
feature_names = [c for c in df_features.columns if c != TARGET_COLUMN]

# Explained variance analysis
plot_explained_variance(X_all)

In [ ]:
# Apply PCA (2 components for visualization)
X_pca, pca, pca_scaler = apply_pca(X_all, n_components=2)

In [ ]:
# Find optimal number of clusters
K_range, inertias, silhouettes = find_optimal_clusters(X_pca, max_k=8)

In [ ]:
# Apply K-Means and visualize with red team overlay
labels, kmeans = apply_kmeans(X_pca, n_clusters=3)
plot_pca_clusters(X_pca, labels, suspicious=y_all)

In [ ]:
# Analyze cluster composition
cluster_df = pd.DataFrame({'cluster': labels, 'is_suspicious': y_all})
print("Cluster composition:")
print(cluster_df.groupby('cluster')['is_suspicious'].agg(['count', 'sum', 'mean']))
print("\n'mean' = proportion of red team events in each cluster")

## 6. Time Series Analysis (Module 10)

Model temporal trends in login behavior and detect sudden deviations.

In [ ]:
# Aggregate to hourly time series
ts = aggregate_login_timeseries(df, freq_seconds=3600)
print(f"Time series: {len(ts)} hourly bins")
print(f"Mean events/hour: {ts.mean():.0f}")
print(f"Max events/hour: {ts.max():.0f}")

In [ ]:
# Seasonal decomposition (daily seasonality = 24 hours)
decomposition = decompose_timeseries(ts, period=24)

In [ ]:
# Detect temporal anomalies using z-score
anomalies = detect_temporal_anomalies(ts, window=24, threshold=3.0)
plot_timeseries_with_anomalies(ts, anomalies)

## 7. Model Selection and Regularization (Module 9 / Module 15)

Compare multiple models using cross-validation.
Tune regularization strength for logistic regression.
Module 15 (Gradient Descent) is reflected in solver/convergence tuning.

In [ ]:
# Prepare train/test split
X_train, X_test, y_train, y_test, scaler, feat_names = prepare_train_test(df_features)

In [ ]:
# Compare all candidate models
print("Cross-validation model comparison:")
results = compare_models(X_train, y_train, cv=5)

In [ ]:
# Tune regularization (C parameter) for logistic regression
best_lr, reg_results = tune_regularization(X_train, y_train, cv=5)

In [ ]:
# Learning curve for the tuned logistic regression
plot_learning_curve(best_lr, X_train, y_train, 'Logistic Regression (Tuned)')

## 8. Classification: KNN and Logistic Regression (Modules 11 & 13)

Build baseline classifiers:

- **KNN** captures local behavioral similarities
- **Logistic Regression** provides interpretable feature weights

In [ ]:
# Find optimal k for KNN
best_k, k_scores = find_optimal_k(X_train, y_train, k_range=range(1, 16), cv=5)

In [ ]:
# Train KNN with optimal k
knn = train_knn(X_train, y_train, n_neighbors=best_k)
print_classification_report(knn, X_test, y_test, 'KNN')

In [ ]:
# Train Logistic Regression with feature coefficient analysis
lr = train_logistic_regression(X_train, y_train, feature_names=feat_names, C=best_lr.C)
print_classification_report(lr, X_test, y_test, 'Logistic Regression')

In [ ]:
# ROC curves and confusion matrices
models = {'KNN': knn, 'Logistic Regression': lr}
plot_roc_curves(models, X_test, y_test)
plot_confusion_matrices(models, X_test, y_test)

## 9. Decision Trees (Module 14)

Build interpretable rule sets that security analysts can act on directly.
Example rule: *"if failed_logins > 5 AND new_IP AND off_hours → flag as suspicious"*

In [ ]:
# Tune tree depth
best_depth, depth_results = tune_tree_depth(X_train, y_train, max_depth_range=range(2, 16))

In [ ]:
# Train decision tree with optimal depth
dt = train_decision_tree(X_train, y_train, max_depth=best_depth)

# Visualize the tree
visualize_tree(dt, feat_names)

In [ ]:
# Extract human-readable rules
rules = extract_rules(dt, feat_names, max_depth=4)

In [ ]:
# Feature importance from the decision tree
importance_df = get_feature_importance(dt, feat_names)

In [ ]:
# Evaluate decision tree on test set
print_section('Decision Tree \u2014 Test Set Evaluation')
evaluate_model(dt, X_test, y_test)

## 10. Final Model Comparison

In [ ]:
# Compare all models on the test set
all_models = {
    'KNN': knn,
    'Logistic Regression': lr,
    'Decision Tree': dt,
}

for name, model in all_models.items():
    print_section(name)
    evaluate_model(model, X_test, y_test)

# Combined ROC curves
plot_roc_curves(all_models, X_test, y_test)

In [ ]:
# Save the best model
save_model(best_lr, 'best_logistic_regression.joblib')
save_model(dt, 'best_decision_tree.joblib')
print("\nModels saved to models/ directory.")

## 11. Conclusions and Next Steps

### Key Findings

- *Fill in after running the analysis:*
- Which model achieved the highest ROC AUC?
- Which features were most important for detecting anomalies?
- Did PCA clustering reveal natural groupings that correlate with red team activity?
- Were temporal anomalies aligned with known red team event times?

### Interpretable Rules for Security Analysts

- The decision tree produces rules like:
  *"If login_count_1h > X AND is_new_dest_computer = 1 AND is_off_hours = 1 → Suspicious"*
- These can be directly implemented as alert rules in a SIEM system.

### Limitations

- The LANL dataset is anonymized — real-world features like IP geolocation and user-agent strings are not available.
- Red team events are extremely rare (severe class imbalance), which affects model training.
- We used a sample of the full dataset; results may differ at full scale.

### Future Work

- Scale to the full 1.6B event dataset using chunked processing or Dask.
- Experiment with ensemble methods (Gradient Boosting, XGBoost).
- Incorporate network flow data (`flows.txt.gz`) and DNS data (`dns.txt.gz`) for multi-source analysis.
- Explore deep learning approaches (autoencoders for anomaly detection).

### Citation

A. D. Kent, "Comprehensive, Multi-Source Cybersecurity Events,"
Los Alamos National Laboratory, http://dx.doi.org/10.17021/1179829, 2015.